# LeetCode 2162 — Minimum Cost to Set Cooking Time bằng Python

Link đề: https://leetcode.com/problems/minimum-cost-to-set-cooking-time/ · Độ khó: **Medium** · Chủ đề: Math, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def minCostSetTime(self, startAt: int, moveCost: int, pushCost: int, targetSeconds: int) -> int:
        ...
```

Bài **liệt kê có suy luận**: không gian tìm kiếm nhỏ (tối đa `10 + 100 + 1000 + 10000` dãy phím), nhưng phân tích kỹ cho thấy chỉ có **tối đa 2 ứng viên** đáng xét. Notebook đi từ **thử mọi dãy 1–4 chữ số** → **duyệt số phút `m`** → **hai ứng viên `(m, s)` và `(m−1, s+60)`** → **chứng minh bỏ số 0 đầu luôn tốt hơn** → **mô hình đồ thị / DP trên trạng thái ngón tay** (để mở rộng cho bàn phím tổng quát).

## 1. Mục tiêu học

1. Mô hình hóa chính xác cách lò vi sóng **chuẩn hóa** dãy phím: thêm số 0 bên trái đủ 4 chữ số, 2 chữ số đầu là phút, 2 chữ số cuối là giây, **giây có thể ≥ 60** (tới 99).
2. Viết hàm **chi phí** của một dãy phím: mỗi lần **đổi** chữ số dưới ngón tay tốn `moveCost`, mỗi lần bấm tốn `pushCost`.
3. Suy ra chỉ cần xét hai cách biểu diễn `(m, s)` với `s ∈ [0, 99]`, `m ∈ [0, 99]`: `(t // 60, t % 60)` và `(t // 60 − 1, t % 60 + 60)`.
4. Chứng minh **bỏ các số 0 đứng đầu** không bao giờ làm chi phí tăng.
5. Kiểm thử toàn diện bằng **vét cạn mọi dãy phím** trên mọi `targetSeconds` (chỉ 6039 giá trị!).

## 2. Đề bài & ví dụ

Lò vi sóng hỗ trợ thời gian nấu từ **1 giây** đến **99 phút 99 giây**. Để đặt giờ, bạn bấm **tối đa 4 chữ số**. Lò thêm số 0 bên trái cho đủ 4 chữ số, coi 2 chữ số đầu là **phút**, 2 chữ số cuối là **giây**, rồi **cộng lại** thành thời gian nấu. Ví dụ:

- bấm `9 5 4` → `0954` → 9 phút 54 giây;
- bấm `0 0 0 8` → 0 phút 8 giây;
- bấm `8 0 9 0` → 80 phút 90 giây;
- bấm `8 1 3 0` → 81 phút 30 giây.

Ban đầu ngón tay ở chữ số `startAt`. **Di chuyển** ngón tay tới một chữ số **khác** tốn `moveCost`; **bấm** chữ số dưới ngón tay một lần tốn `pushCost`. Trả về chi phí nhỏ nhất để đặt đúng `targetSeconds` giây.

**Ví dụ 1:** `startAt = 1, moveCost = 2, pushCost = 1, targetSeconds = 600` → `6` (bấm `1 0 0 0`: `1 + 2 + 1 + 1 + 1`).

**Ví dụ 2:** `startAt = 0, moveCost = 1, pushCost = 2, targetSeconds = 76` → `6` (bấm `7 6`: `1 + 2 + 1 + 2`).

**Ràng buộc:** `0 ≤ startAt ≤ 9`, `1 ≤ moveCost, pushCost ≤ 10⁵`, `1 ≤ targetSeconds ≤ 6039`.

In [1]:
import random
import time
from itertools import product
from typing import List

examples = [((1, 2, 1, 600), 6), ((0, 1, 2, 76), 6)]

def interpret(keys: str) -> int:
    """Thời gian (giây) mà lò hiểu từ dãy phím (1..4 chữ số)."""
    s = keys.rjust(4, "0")
    return int(s[:2]) * 60 + int(s[2:])

for k in ["954", "0008", "8090", "8130", "1000", "960", "76", "116"]:
    print(f"bấm {k:>4} -> chuẩn hóa {k.rjust(4, '0')} -> {interpret(k)} giây")

bấm  954 -> chuẩn hóa 0954 -> 594 giây
bấm 0008 -> chuẩn hóa 0008 -> 8 giây
bấm 8090 -> chuẩn hóa 8090 -> 4890 giây
bấm 8130 -> chuẩn hóa 8130 -> 4890 giây
bấm 1000 -> chuẩn hóa 1000 -> 600 giây
bấm  960 -> chuẩn hóa 0960 -> 600 giây
bấm   76 -> chuẩn hóa 0076 -> 76 giây
bấm  116 -> chuẩn hóa 0116 -> 76 giây


## 3. Kiến thức nền

### 3.1 `divmod` — tách phút và giây

In [2]:
for t in [600, 76, 6039, 5999, 59]:
    print(t, "->", divmod(t, 60))

600 -> (10, 0)
76 -> (1, 16)
6039 -> (100, 39)
5999 -> (99, 59)
59 -> (0, 59)


### 3.2 Định dạng số có số 0 bên trái và bỏ số 0 đầu

`f"{m:02d}{s:02d}"` cho chuỗi 4 ký tự; `str.lstrip("0")` bỏ **các số 0 đứng đầu** (không phải mọi số 0!).

In [3]:
m, s = 10, 0
four = f"{m:02d}{s:02d}"
print(repr(four), "->", repr(four.lstrip("0")), "| sai lầm thường gặp: replace ->", repr(four.replace("0", "")))

'1000' -> '1000' | sai lầm thường gặp: replace -> '1'


### 3.3 Hàm chi phí của một dãy phím

Ngón tay ở `prev` (ban đầu `startAt`). Với mỗi chữ số `d`: **nếu** `d ≠ prev` **thì** cộng `moveCost`; luôn cộng `pushCost`; đặt `prev = d`.

In [4]:
def cost_of(keys: str, startAt: int, moveCost: int, pushCost: int) -> int:
    total, prev = 0, startAt
    for ch in keys:
        d = int(ch)
        if d != prev:
            total += moveCost
        total += pushCost
        prev = d
    return total

print("ví dụ 1, '1000':", cost_of("1000", 1, 2, 1), "| '0960':", cost_of("0960", 1, 2, 1), "| '960':", cost_of("960", 1, 2, 1))
print("ví dụ 2, '76':", cost_of("76", 0, 1, 2), "| '0076':", cost_of("0076", 0, 1, 2), "| '116':", cost_of("116", 0, 1, 2))

ví dụ 1, '1000': 6 | '0960': 12 | '960': 9
ví dụ 2, '76': 6 | '0076': 10 | '116': 8


### 3.4 `itertools.product` — sinh mọi dãy chữ số

In [5]:
print(["".join(p) for p in product("01", repeat=3)])
print("số dãy 1..4 chữ số:", sum(10 ** L for L in range(1, 5)))

['000', '001', '010', '011', '100', '101', '110', '111']
số dãy 1..4 chữ số: 11110


## 4. Ý tưởng nền tảng / trực giác

Một thời gian `t` có thể biểu diễn là `60·m + s` với `0 ≤ m ≤ 99`, `0 ≤ s ≤ 99`. Vì `s ≤ 99 < 120`, với mỗi `m` chỉ có tối đa một `s`, và `s = t − 60m ∈ [0, 99]` ⇒ `m ∈ [(t − 99)/60, t/60]` — khoảng dài `99/60 < 2` ⇒ **tối đa 2 giá trị `m`**:

- `m₁ = t // 60`, `s₁ = t % 60` (luôn có `s₁ ≤ 59`);
- `m₂ = m₁ − 1`, `s₂ = s₁ + 60` (hợp lệ khi `m₂ ≥ 0` và `s₂ ≤ 99`, tức `s₁ ≤ 39`).

Thêm ràng buộc `m ≤ 99`: với `t ≥ 6000`, `m₁ = 100` **không** hợp lệ, chỉ còn `m₂`.

Tính tay ví dụ 2 (`t = 76`): `m₁ = 1, s₁ = 16` → bấm `116` (bỏ số 0 đầu của `0116`); `m₂ = 0, s₂ = 76` → bấm `76`. Chi phí: `116`: di chuyển 0→1 (1), bấm (2), bấm 1 (2), di chuyển 1→6 (1), bấm (2) = 8; `76`: 6 ⇒ đáp án 6.

In [6]:
def candidates(t):
    out = []
    m, s = divmod(t, 60)
    for mm, ss in [(m, s), (m - 1, s + 60)]:
        if 0 <= mm <= 99 and 0 <= ss <= 99:
            out.append((mm, ss))
    return out

for t in [600, 76, 6039, 5999, 59, 100, 140]:
    print(f"t={t:>4}: ứng viên (phút, giây) = {candidates(t)}")

t= 600: ứng viên (phút, giây) = [(10, 0), (9, 60)]
t=  76: ứng viên (phút, giây) = [(1, 16), (0, 76)]
t=6039: ứng viên (phút, giây) = [(99, 99)]
t=5999: ứng viên (phút, giây) = [(99, 59)]
t=  59: ứng viên (phút, giây) = [(0, 59)]
t= 100: ứng viên (phút, giây) = [(1, 40)]
t= 140: ứng viên (phút, giây) = [(2, 20), (1, 80)]


## 5. Cách 1 — Brute Force / baseline: thử mọi dãy phím 1–4 chữ số

Sinh `11 110` dãy, giữ dãy có `interpret(keys) == t`, lấy chi phí nhỏ nhất. Không cần suy luận gì — **chuẩn đối chiếu** hoàn hảo.

In [7]:
ALL_KEYS = ["".join(p) for L in range(1, 5) for p in product("0123456789", repeat=L)]
BY_TIME = {}
for k in ALL_KEYS:
    BY_TIME.setdefault(interpret(k), []).append(k)

def min_cost_brute(startAt, moveCost, pushCost, t):
    return min(cost_of(k, startAt, moveCost, pushCost) for k in BY_TIME[t])

for args, exp in examples:
    print(min_cost_brute(*args), "(kỳ vọng", exp, ")", "| các dãy cho t =", args[3], ":", BY_TIME[args[3]])

6 (kỳ vọng 6 ) | các dãy cho t = 600 : ['960', '0960', '1000']
6 (kỳ vọng 6 ) | các dãy cho t = 76 : ['76', '076', '116', '0076', '0116']


**Độ phức tạp:** sinh bảng một lần `O(10⁴ · 4)`; mỗi truy vấn `O(số dãy cho t · 4)` — rất nhỏ (tối đa vài chục dãy). Thực ra brute force ở đây **đủ nhanh để nộp**, nhưng ta muốn hiểu cấu trúc bài.

**Nút thắt:** phần lớn dãy chỉ khác nhau ở **số 0 đầu** (`76`, `076`, `0076`) — phân tích để bỏ chúng.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (tối đa 2 biểu diễn):** đã chứng minh ở mục 4.

**Insight 2 (bỏ số 0 đầu không bao giờ thiệt):** xét dãy `0 d₁ d₂ …` (bắt đầu bằng `0`) và dãy `d₁ d₂ …`. Cả hai cho **cùng thời gian** (lò tự thêm 0). Chi phí dãy có `0` đầu = (di chuyển `startAt → 0` nếu khác) + `pushCost` + (di chuyển `0 → d₁` nếu khác) + phần còn lại. Chi phí dãy bỏ `0` = (di chuyển `startAt → d₁` nếu khác) + phần còn lại. Vì "có di chuyển `startAt → d₁`" ≤ "có di chuyển `startAt → 0`" + "có di chuyển `0 → d₁`" (nếu `startAt ≠ d₁` thì ít nhất một trong hai bước kia là di chuyển thật), và còn tiết kiệm `pushCost > 0` ⇒ bỏ `0` **luôn tốt hơn nghiêm ngặt**.

Lặp lại ⇒ với mỗi biểu diễn `(m, s)`, dãy tốt nhất là `f"{m:02d}{s:02d}".lstrip("0")` (không rỗng vì `t ≥ 1`).

**Quy tắc:** **nếu** `(m₁, s₁)` hợp lệ **thì** tính chi phí; **nếu** `(m₁ − 1, s₁ + 60)` hợp lệ **thì** tính chi phí; trả min.

In [8]:
# Kiểm chứng Insight 2 bằng vét cạn: mọi dãy bắt đầu bằng '0' đều đắt hơn dãy bỏ '0' đó
viol = 0
for k in ALL_KEYS:
    if k[0] == "0" and len(k) > 1:
        for st in range(10):
            for mc, pc in [(1, 1), (5, 1), (1, 5), (100000, 1)]:
                if cost_of(k, st, mc, pc) <= cost_of(k[1:], st, mc, pc):
                    viol += 1
print("số trường hợp bỏ '0' đầu KHÔNG làm rẻ hơn:", viol)

số trường hợp bỏ '0' đầu KHÔNG làm rẻ hơn: 0


### 6.1 Thống kê: bao nhiêu `t` có hai ứng viên, và ứng viên thứ hai thắng khi nào?

Ứng viên 2 tồn tại ⇔ `m₁ ≥ 1` và `s₁ ≤ 39` (hoặc `t ≥ 6000`). Ứng viên 2 thường **dài bằng hoặc ngắn hơn** khi `m₁ = 1` (ví dụ `0:76` → `76` chỉ 2 chữ số so với `116`), hoặc khi các chữ số của nó "ít đổi" hơn (ví dụ `9:60` khi ngón tay đang ở `9`). Cell dưới đếm trên mọi `t` với tham số `startAt = 0, moveCost = pushCost = 1`:

In [9]:
one = two = second_wins = tie = 0
examples_second = []
for t in range(1, 6040):
    cand = candidates(t)
    if len(cand) == 1:
        one += 1; continue
    two += 1
    c = [cost_of(f"{m:02d}{s:02d}".lstrip("0"), 0, 1, 1) for m, s in cand]
    if c[1] < c[0]:
        second_wins += 1
        if len(examples_second) < 6:
            examples_second.append((t, cand[1], f"{cand[1][0]:02d}{cand[1][1]:02d}".lstrip("0")))
    elif c[1] == c[0]:
        tie += 1
print(f"t có 1 ứng viên: {one}, có 2 ứng viên: {two}; ứng viên 2 rẻ hơn hẳn: {second_wins}, hòa: {tie}")
print("vài t mà ứng viên 2 thắng:", examples_second)

t có 1 ứng viên: 2079, có 2 ứng viên: 3960; ứng viên 2 rẻ hơn hẳn: 900, hòa: 2211
vài t mà ứng viên 2 thắng: [(60, (0, 60), '60'), (61, (0, 61), '61'), (62, (0, 62), '62'), (63, (0, 63), '63'), (64, (0, 64), '64'), (65, (0, 65), '65')]


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Duyệt mọi số phút `m = 0..99`

In [10]:
def min_cost_enum_minutes(startAt, moveCost, pushCost, t):
    best = float("inf")
    for m in range(100):
        s = t - 60 * m
        if 0 <= s <= 99:
            keys = f"{m:02d}{s:02d}".lstrip("0")
            best = min(best, cost_of(keys, startAt, moveCost, pushCost))
    return best

for args, exp in examples:
    print(min_cost_enum_minutes(*args), "(kỳ vọng", exp, ")")

6 (kỳ vọng 6 )
6 (kỳ vọng 6 )


**Độ phức tạp:** `O(100)` — hằng số. Không cần nhớ công thức hai ứng viên, chỉ cần biết `m, s ∈ [0, 99]`.

### 7.2 Cách 3 — Hai ứng viên `(m, s)` và `(m − 1, s + 60)`

In [11]:
def min_cost_two(startAt, moveCost, pushCost, t):
    def f(m, s):
        if not (0 <= m <= 99 and 0 <= s <= 99):
            return float("inf")
        return cost_of(f"{m:02d}{s:02d}".lstrip("0"), startAt, moveCost, pushCost)
    m, s = divmod(t, 60)
    return min(f(m, s), f(m - 1, s + 60))

for args, exp in examples:
    print(min_cost_two(*args), "(kỳ vọng", exp, ")")

6 (kỳ vọng 6 )
6 (kỳ vọng 6 )


**Độ phức tạp:** `O(1)`.

### 7.3 Cách 4 — Tính chi phí trực tiếp trên chữ số (không dùng chuỗi)

Tránh tạo chuỗi: tách 4 chữ số `[m // 10, m % 10, s // 10, s % 10]`, bỏ số 0 đầu bằng chỉ số `i`, cộng chi phí. Cách viết này dịch thẳng sang C++/Java.

In [12]:
def min_cost_digits(startAt, moveCost, pushCost, t):
    best = None
    m, s = divmod(t, 60)
    for mm, ss in ((m, s), (m - 1, s + 60)):
        if mm < 0 or mm > 99 or ss > 99:
            continue
        digits = (mm // 10, mm % 10, ss // 10, ss % 10)
        i = 0
        while digits[i] == 0:              # t ≥ 1 nên không chạy quá 3
            i += 1
        c, prev = 0, startAt
        for d in digits[i:]:
            c += pushCost + (moveCost if d != prev else 0)
            prev = d
        best = c if best is None else min(best, c)
    return best

for args, exp in examples:
    print(min_cost_digits(*args), "(kỳ vọng", exp, ")")

6 (kỳ vọng 6 )
6 (kỳ vọng 6 )


### 7.4 Cách 5 — Mô hình "ngón tay" như DP trên trạng thái (tổng quát hóa)

Nếu bàn phím **không** có chi phí di chuyển đồng đều (ví dụ chi phí là khoảng cách Manhattan trên bàn phím 3×4), ta không suy luận "bỏ 0" dễ như trên. Cách tổng quát: với mỗi dãy chữ số **cố định** `d₁…d_L` (biểu diễn ứng viên, **có thể** có 0 đầu), chi phí là `Σ dist(prev, dᵢ) + L·pushCost`. Khi số dãy ứng viên nhỏ (≤ vài chục), ta chỉ cần liệt kê các ứng viên + mọi cách thêm số 0 đầu, rồi tính bằng hàm `dist` bất kỳ. Ở đây `dist(a, b) = moveCost·[a ≠ b]`.

In [13]:
def min_cost_general(startAt, t, dist, pushCost):
    """dist(a, b): chi phí di chuyển ngón tay từ a sang b. Xét mọi biểu diễn + mọi số 0 đầu."""
    best = float("inf")
    for m in range(100):
        s = t - 60 * m
        if 0 <= s <= 99:
            four = f"{m:02d}{s:02d}"
            for cut in range(4):                    # giữ lại four[cut:], miễn là bỏ đi toàn số 0
                if four[:cut].strip("0") == "" and four[cut:]:
                    c, prev = 0, startAt
                    for ch in four[cut:]:
                        d = int(ch); c += dist(prev, d) + pushCost; prev = d
                    best = min(best, c)
    return best

uniform = lambda mc: (lambda a, b: mc if a != b else 0)
for (st, mc, pc, t), exp in examples:
    print(min_cost_general(st, t, uniform(mc), pc), "(kỳ vọng", exp, ")")

6 (kỳ vọng 6 )
6 (kỳ vọng 6 )


**Độ phức tạp:** `O(100 · 4 · 4)` — hằng số; mở rộng được cho mọi hàm `dist` (bài tập 5).

## 8. TRACE từng bước

### 8.1 Trace cách 3 trên hai ví dụ của đề

In [14]:
def trace_two(startAt, moveCost, pushCost, t):
    m, s = divmod(t, 60)
    print(f"  t = {t}: divmod -> (m, s) = ({m}, {s})")
    best = float("inf")
    for mm, ss in [(m, s), (m - 1, s + 60)]:
        if not (0 <= mm <= 99 and 0 <= ss <= 99):
            print(f"    ứng viên ({mm}, {ss}): không hợp lệ"); continue
        keys = f"{mm:02d}{ss:02d}".lstrip("0")
        steps, total, prev = [], 0, startAt
        for ch in keys:
            d = int(ch)
            if d != prev:
                steps.append(f"di {prev}->{d}(+{moveCost})"); total += moveCost
            steps.append(f"bấm {d}(+{pushCost})"); total += pushCost
            prev = d
        print(f"    ứng viên ({mm}, {ss}) -> bấm '{keys}': " + ", ".join(steps) + f" = {total}")
        best = min(best, total)
    print("  => min =", best)
    return best

for args, _ in examples:
    trace_two(*args)

  t = 600: divmod -> (m, s) = (10, 0)
    ứng viên (10, 0) -> bấm '1000': bấm 1(+1), di 1->0(+2), bấm 0(+1), bấm 0(+1), bấm 0(+1) = 6
    ứng viên (9, 60) -> bấm '960': di 1->9(+2), bấm 9(+1), di 9->6(+2), bấm 6(+1), di 6->0(+2), bấm 0(+1) = 9
  => min = 6
  t = 76: divmod -> (m, s) = (1, 16)
    ứng viên (1, 16) -> bấm '116': di 0->1(+1), bấm 1(+2), bấm 1(+2), di 1->6(+1), bấm 6(+2) = 8
    ứng viên (0, 76) -> bấm '76': di 0->7(+1), bấm 7(+2), di 7->6(+1), bấm 6(+2) = 6
  => min = 6


### 8.2 Trace các trường hợp biên của ứng viên (ví dụ tự dựng — đã đối chiếu vét cạn, đối chiếu lại đề gốc)

In [15]:
for args in [(0, 1, 1, 6039), (9, 3, 1, 6000), (5, 1, 1, 99), (1, 10, 1, 60), (8, 1, 100, 5999)]:
    r = trace_two(*args)
    assert r == min_cost_brute(*args)
    print()

  t = 6039: divmod -> (m, s) = (100, 39)
    ứng viên (100, 39): không hợp lệ
    ứng viên (99, 99) -> bấm '9999': di 0->9(+1), bấm 9(+1), bấm 9(+1), bấm 9(+1), bấm 9(+1) = 5
  => min = 5

  t = 6000: divmod -> (m, s) = (100, 0)
    ứng viên (100, 0): không hợp lệ
    ứng viên (99, 60) -> bấm '9960': bấm 9(+1), bấm 9(+1), di 9->6(+3), bấm 6(+1), di 6->0(+3), bấm 0(+1) = 10
  => min = 10

  t = 99: divmod -> (m, s) = (1, 39)
    ứng viên (1, 39) -> bấm '139': di 5->1(+1), bấm 1(+1), di 1->3(+1), bấm 3(+1), di 3->9(+1), bấm 9(+1) = 6
    ứng viên (0, 99) -> bấm '99': di 5->9(+1), bấm 9(+1), bấm 9(+1) = 3
  => min = 3

  t = 60: divmod -> (m, s) = (1, 0)
    ứng viên (1, 0) -> bấm '100': bấm 1(+1), di 1->0(+10), bấm 0(+1), bấm 0(+1) = 13
    ứng viên (0, 60) -> bấm '60': di 1->6(+10), bấm 6(+1), di 6->0(+10), bấm 0(+1) = 22
  => min = 13

  t = 5999: divmod -> (m, s) = (99, 59)
    ứng viên (99, 59) -> bấm '9959': di 8->9(+1), bấm 9(+100), bấm 9(+100), di 9->5(+1), bấm 5(+100), di 5->9(+1

## 9. Code nộp LeetCode

In [16]:
class Solution:
    def minCostSetTime(self, startAt: int, moveCost: int, pushCost: int, targetSeconds: int) -> int:
        def cost(m: int, s: int) -> float:
            if not (0 <= m <= 99 and 0 <= s <= 99):
                return float("inf")
            keys = f"{m:02d}{s:02d}".lstrip("0")      # bỏ số 0 đầu: luôn rẻ hơn
            total, prev = 0, startAt
            for ch in keys:
                d = ord(ch) - 48
                if d != prev:
                    total += moveCost
                total += pushCost
                prev = d
            return total

        m, s = divmod(targetSeconds, 60)
        return int(min(cost(m, s), cost(m - 1, s + 60)))

for args, exp in examples:
    assert Solution().minCostSetTime(*args) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

1. **Mọi dãy phím hợp lệ** cho `t` có dạng (một số số 0) + chuỗi chuẩn của một cặp `(m, s)` với `60m + s = t`, `0 ≤ m, s ≤ 99` — vì lò chỉ đọc 4 chữ số sau khi thêm 0 trái.
2. **Chỉ có ≤ 2 cặp** `(m, s)` (mục 4): `s = t − 60m ∈ [0, 99]` buộc `m ∈ {t // 60, t // 60 − 1}`; ràng buộc `m ≤ 99` loại `m = 100` khi `t ≥ 6000`.
3. **Trong các dãy của cùng một cặp**, dãy bỏ hết số 0 đầu là rẻ nhất (Insight 2, chứng minh bằng bất đẳng thức tam giác của "có di chuyển hay không" + tiết kiệm `pushCost`; đã kiểm bằng vét cạn ở mục 6).
4. ⇒ Lấy min trên ≤ 2 dãy là tối ưu toàn cục. Mục 12 kiểm **mọi** `t ∈ [1, 6039]` so với vét cạn cho nhiều bộ tham số.

## 11. Phân tích Time / Space Complexity

| Cách | Time mỗi truy vấn | Space |
|---|---|---|
| 1. Vét cạn mọi dãy | `O(số dãy của t)` (+ tiền xử lý `O(10⁴)`) | `O(10⁴)` |
| 2. Duyệt `m = 0..99` | `O(100)` | `O(1)` |
| 3. Hai ứng viên | `O(1)` | `O(1)` |
| 4. Chữ số trực tiếp | `O(1)` | `O(1)` |
| 5. Tổng quát hàm `dist` | `O(100 · 16)` | `O(1)` |

In [17]:
rnd = random.Random(2162)
qs = [(rnd.randint(0, 9), rnd.randint(1, 10**5), rnd.randint(1, 10**5), rnd.randint(1, 6039)) for _ in range(20000)]
for name, fn in [("brute", min_cost_brute), ("enum_minutes", min_cost_enum_minutes), ("two", min_cost_two),
                 ("digits", min_cost_digits), ("Solution", Solution().minCostSetTime)]:
    t0 = time.perf_counter()
    for q in qs:
        fn(*q)
    print(f"{name:<13}: 20000 truy vấn trong {(time.perf_counter() - t0) * 1000:7.1f} ms")

brute        : 20000 truy vấn trong    18.4 ms


enum_minutes : 20000 truy vấn trong    99.2 ms


two          : 20000 truy vấn trong    26.9 ms


digits       : 20000 truy vấn trong    12.4 ms


Solution     : 20000 truy vấn trong    24.2 ms


## 12. Bộ test

In [18]:
ALL = {
    "enum_minutes": min_cost_enum_minutes,
    "two": min_cost_two,
    "digits": min_cost_digits,
    "general": lambda st, mc, pc, t: min_cost_general(st, t, uniform(mc), pc),
    "Solution": lambda st, mc, pc, t: Solution().minCostSetTime(st, mc, pc, t),
}

def check(startAt, moveCost, pushCost, t, exp=None):
    ref = min_cost_brute(startAt, moveCost, pushCost, t)
    for name, fn in ALL.items():
        got = fn(startAt, moveCost, pushCost, t)
        assert got == ref, (name, startAt, moveCost, pushCost, t, got, ref)
    if exp is not None:
        assert ref == exp
    return ref

for args, exp in examples:
    check(*args, exp)
print("Ví dụ đề: OK")

check(0, 1, 1, 1, 2)            # bấm '1': di 0->1 + bấm
check(1, 1, 1, 1, 1)            # ngón tay sẵn ở 1
check(9, 1, 1, 6039, 4)         # chỉ (99, 99): bấm '9999', ngón tay sẵn ở 9 -> 4 lần bấm
print("Edge case cơ bản: OK")

Ví dụ đề: OK
Edge case cơ bản: OK


Các edge case tiếp theo là **ví dụ tự dựng**: kỳ vọng được lấy từ vét cạn (không tính nhẩm) rồi mới ghi cứng — quy tắc an toàn để không assert sai. In kèm danh sách mọi dãy phím hợp lệ để người đọc tự kiểm (đối chiếu lại đề gốc).

In [19]:
edge = [((0, 1, 1, 1), 2), ((1, 1, 1, 1), 1), ((9, 1, 1, 6039), 4), ((0, 1, 1, 6000), None),
        ((5, 100000, 100000, 6039), None), ((3, 7, 2, 60), None), ((0, 1, 1, 99), None)]
for args, exp in edge:
    r = check(*args, exp)
    print(f"  {args} -> {r}  (dãy: {BY_TIME[args[3]]})")
print("Edge case (kỳ vọng lấy từ vét cạn): OK")

rnd = random.Random(2021)
for t in range(1, 6040):                          # MỌI targetSeconds
    check(rnd.randint(0, 9), rnd.randint(1, 10**5), rnd.randint(1, 10**5), t)
print("Mọi t ∈ [1, 6039] với tham số ngẫu nhiên: OK")

for _ in range(5000):
    check(rnd.randint(0, 9), rnd.randint(1, 5), rnd.randint(1, 5), rnd.randint(1, 6039))
print("5000 bộ tham số nhỏ (nhiều hòa chi phí): OK")

  (0, 1, 1, 1) -> 2  (dãy: ['1', '01', '001', '0001'])
  (1, 1, 1, 1) -> 1  (dãy: ['1', '01', '001', '0001'])
  (9, 1, 1, 6039) -> 4  (dãy: ['9999'])
  (0, 1, 1, 6000) -> 7  (dãy: ['9960'])
  (5, 100000, 100000, 6039) -> 500000  (dãy: ['9999'])
  (3, 7, 2, 60) -> 18  (dãy: ['60', '060', '100', '0060', '0100'])
  (0, 1, 1, 99) -> 3  (dãy: ['99', '099', '139', '0099', '0139'])
Edge case (kỳ vọng lấy từ vét cạn): OK


Mọi t ∈ [1, 6039] với tham số ngẫu nhiên: OK


5000 bộ tham số nhỏ (nhiều hòa chi phí): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ xét `(t // 60, t % 60)`, quên `(m − 1, s + 60)`

In [20]:
def bug_one_candidate(startAt, moveCost, pushCost, t):
    m, s = divmod(t, 60)
    if m > 99:
        return float("inf")
    return cost_of(f"{m:02d}{s:02d}".lstrip("0"), startAt, moveCost, pushCost)

args = (0, 1, 2, 76)
print("sai :", bug_one_candidate(*args), "| đúng:", Solution().minCostSetTime(*args))

sai : 8 | đúng: 6


**Vì sao sai:** `76` giây bấm `76` (0 phút 76 giây) rẻ hơn `116`. Ở `t = 6039` lỗi còn tệ hơn: trả `inf`.

### Lỗi 2 — Quên giới hạn `m ≤ 99`

In [21]:
def bug_no_minute_cap(startAt, moveCost, pushCost, t):
    best = float("inf")
    m, s = divmod(t, 60)
    for mm, ss in [(m, s), (m - 1, s + 60)]:
        if mm >= 0 and ss <= 99:                       # SAI: thiếu mm <= 99
            best = min(best, cost_of(f"{mm:02d}{ss:02d}".lstrip("0"), startAt, moveCost, pushCost))
    return best

args = (1, 5, 1, 6000)
print("sai :", bug_no_minute_cap(*args), "| đúng:", Solution().minCostSetTime(*args))
print("   (ứng viên sai '10000' có 5 chữ số — lò không cho bấm quá 4 chữ số)")

sai : 10 | đúng: 19
   (ứng viên sai '10000' có 5 chữ số — lò không cho bấm quá 4 chữ số)


**Sửa:** `0 ≤ m ≤ 99`.

### Lỗi 3 — Luôn bấm đủ 4 chữ số (không bỏ số 0 đầu)

In [22]:
def bug_keep_zeros(startAt, moveCost, pushCost, t):
    best = float("inf")
    m, s = divmod(t, 60)
    for mm, ss in [(m, s), (m - 1, s + 60)]:
        if 0 <= mm <= 99 and 0 <= ss <= 99:
            best = min(best, cost_of(f"{mm:02d}{ss:02d}", startAt, moveCost, pushCost))   # SAI
    return best

args = (0, 1, 2, 76)
print("sai :", bug_keep_zeros(*args), "| đúng:", Solution().minCostSetTime(*args))

sai : 10 | đúng: 6


**Vì sao sai:** `0076` tốn thêm hai lần bấm `0`. **Sửa:** `.lstrip("0")`.

### Lỗi 4 — Bỏ **mọi** số 0 bằng `replace`

In [23]:
def bug_replace_zeros(startAt, moveCost, pushCost, t):
    best = float("inf")
    m, s = divmod(t, 60)
    for mm, ss in [(m, s), (m - 1, s + 60)]:
        if 0 <= mm <= 99 and 0 <= ss <= 99:
            best = min(best, cost_of(f"{mm:02d}{ss:02d}".replace("0", ""), startAt, moveCost, pushCost))  # SAI
    return best

args = (1, 2, 1, 600)
print("sai :", bug_replace_zeros(*args), "| đúng:", Solution().minCostSetTime(*args), "| '1000' bị biến thành '1' = 1 giây!")

sai : 1 | đúng: 6 | '1000' bị biến thành '1' = 1 giây!


**Vì sao sai:** chuỗi `1000` thành `1` — đặt 1 giây chứ không phải 600 giây. Chi phí tính ra **thấp hơn thật** (sai theo hướng "quá tốt") — kiểu lỗi chỉ bắt được khi đối chiếu brute force.

### Lỗi 5 — Cộng `moveCost` cho mọi chữ số (kể cả khi ngón tay đã ở đúng chỗ)

In [24]:
def bug_always_move(startAt, moveCost, pushCost, t):
    m, s = divmod(t, 60)
    best = float("inf")
    for mm, ss in [(m, s), (m - 1, s + 60)]:
        if 0 <= mm <= 99 and 0 <= ss <= 99:
            keys = f"{mm:02d}{ss:02d}".lstrip("0")
            best = min(best, len(keys) * (moveCost + pushCost))     # SAI
    return best

args = (1, 2, 1, 600)
print("sai :", bug_always_move(*args), "| đúng:", Solution().minCostSetTime(*args))

sai : 9 | đúng: 6


**Sửa:** chỉ cộng `moveCost` khi chữ số **khác** chữ số trước (ban đầu là `startAt`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Không gian trạng thái nhỏ (≤ 10⁴) ⇒ **vét cạn được**; dùng nó làm chuẩn rồi mới tối ưu.
- Biểu diễn "phút:giây" với giây ≤ 99 ⇒ tối đa **2** cách viết.
- Phần đệm (số 0 đầu) mà máy tự thêm ⇒ **không bao giờ bấm** (chứng minh bằng bất đẳng thức tam giác).
- Kiểm tra đủ biên: `t ≥ 6000` (chỉ còn `m = 99`), `s₁ ≥ 40` (không có ứng viên 2), `t < 60`.
- Ràng buộc nhỏ cho phép test **toàn bộ** miền đầu vào (mọi `t`), không chỉ ngẫu nhiên.
- Thống kê ở mục 6.1 cho thấy ứng viên 2 thắng ở **900** giá trị `t` và hòa ở hơn 2000 giá trị — nếu chỉ thử vài ví dụ ngẫu nhiên, lỗi "quên ứng viên 2" rất dễ lọt; đó là lý do kiểm **mọi** `t`.
- Khi đề mô tả một "máy" (lò vi sóng, đồng hồ, bàn phím), hãy viết **hàm mô phỏng máy** (`interpret`) trước — nó vừa giúp hiểu đề, vừa là nền cho vét cạn.
- Chi phí "di chuyển nếu khác" là một metric (thỏa bất đẳng thức tam giác) ⇒ mọi lập luận "bỏ bước trung gian không thiệt" đều dùng được.

## 15. Biến thể của bài

1. **Bàn phím thật 3×4** (`1 2 3 / 4 5 6 / 7 8 9 / _ 0 _`), chi phí di chuyển = khoảng cách Manhattan × `moveCost` — dùng cách 5 với hàm `dist` riêng (bài tập 5). Khi đó "bỏ số 0 đầu" vẫn đúng? (Có — bất đẳng thức tam giác vẫn đúng với khoảng cách Manhattan.)
2. **Đồng hồ hẹn giờ `hh:mm:ss`** với 6 chữ số: nhiều biểu diễn hơn (phút ≤ 99, giây ≤ 99).
3. **Đếm số dãy phím** cho `t` (bài tập 3).
4. **LeetCode 1320 — Minimum Distance to Type a Word Using Two Fingers:** DP trên vị trí ngón tay.
5. **LeetCode 2224 — Minimum Number of Operations to Convert Time.**

In [25]:
KEYPAD = {1: (0, 0), 2: (0, 1), 3: (0, 2), 4: (1, 0), 5: (1, 1), 6: (1, 2), 7: (2, 0), 8: (2, 1), 9: (2, 2), 0: (3, 1)}

def manhattan(mc):
    return lambda a, b: mc * (abs(KEYPAD[a][0] - KEYPAD[b][0]) + abs(KEYPAD[a][1] - KEYPAD[b][1]))

def brute_general(startAt, t, dist, pushCost):
    best = float("inf")
    for k in BY_TIME[t]:
        c, prev = 0, startAt
        for ch in k:
            d = int(ch); c += dist(prev, d) + pushCost; prev = d
        best = min(best, c)
    return best

rnd_v = random.Random(15)
for _ in range(2000):
    st, t, mc, pc = rnd_v.randint(0, 9), rnd_v.randint(1, 6039), rnd_v.randint(1, 5), rnd_v.randint(1, 5)
    assert min_cost_general(st, t, manhattan(mc), pc) == brute_general(st, t, manhattan(mc), pc)
print("bàn phím 3x4, t=600, start=5:", min_cost_general(5, 600, manhattan(1), 1), "| cách 5 khớp vét cạn trên 2000 test")

bàn phím 3x4, t=600, start=5: 9 | cách 5 khớp vét cạn trên 2000 test


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn dãy | thử 11 110 dãy | tiền xử lý `O(10⁴)` | `O(10⁴)` | chuẩn đối chiếu |
| 2. Duyệt phút | `m = 0..99` | `O(100)` | `O(1)` | không cần nhớ công thức |
| 3. Hai ứng viên | `(m, s)`, `(m−1, s+60)` | `O(1)` | `O(1)` | **nộp bài** |
| 4. Chữ số trực tiếp | như 3, không chuỗi | `O(1)` | `O(1)` | C++/Java |
| 5. Tổng quát `dist` | mọi biểu diễn + số 0 đầu | `O(1600)` | `O(1)` | bàn phím tổng quát |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().minCostSetTime(0, 1, 1, 6039)`?

**Bài 2 (tính tay):** `startAt = 4, moveCost = 3, pushCost = 2, targetSeconds = 100`. Liệt kê ứng viên và chi phí.

**Bài 3 (điền TODO):** Hoàn thiện `count_key_sequences(t)` — số dãy phím (1–4 chữ số) đặt được đúng `t` giây, **không** dùng bảng `BY_TIME` (đếm theo công thức: mỗi biểu diễn `(m, s)` đóng góp bao nhiêu dãy?).

**Bài 4 (tự viết lại):** Viết lại cách 3 trả về **cả dãy phím tối ưu** (chuỗi) và kiểm bằng validator: `interpret(keys) == t`, `cost_of(keys) == ` chi phí tối ưu.

**Bài 5 (mini challenge):** Với bàn phím 3×4 (mục 15), `startAt` cho trước, tìm `t ∈ [1, 6039]` có chi phí đặt **lớn nhất** khi `moveCost = pushCost = 1`. In `t` và dãy phím.

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_key_sequences(t):
    # TODO: với mỗi (m, s) hợp lệ, chuỗi 4 ký tự có z số 0 đầu -> đóng góp (z + 1) dãy (giới hạn độ dài >= 1)
    pass

# Bài 4 — TODO
def best_keys(startAt, moveCost, pushCost, t):
    # TODO: như cách 3 nhưng giữ lại chuỗi
    pass

# Bài 5 — TODO
def hardest_time_on_keypad(startAt):
    # TODO: duyệt t, dùng min_cost_general với manhattan(1)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `t ≥ 6000` ⇒ chỉ có `(99, 99)`.
- Mức 2: Bấm `9999`, ngón tay từ 0 phải di chuyển một lần.
- Mức 3: `1 + 4·1 = 5`.

**Bài 2**
- Mức 1: `divmod(100, 60) = (1, 40)` ⇒ ứng viên `140`; `(0, 100)` không hợp lệ (giây > 99).
- Mức 2: `140`: di 4→1, bấm, di 1→4, bấm, di 4→0, bấm.
- Mức 3: `3·3 + 3·2 = 15`.

**Bài 3**
- Mức 1: Chuỗi chuẩn `four` có `z` số 0 đầu.
- Mức 2: Có thể bỏ `0..z` số 0 đầu, nhưng phải còn ≥ 1 ký tự (luôn đúng vì `t ≥ 1` ⇒ `z ≤ 3`).
- Mức 3: Tổng `(z + 1)` trên các biểu diễn; đối chiếu `len(BY_TIME[t])`.

**Bài 4**
- Mức 1: Trả cặp `(cost, keys)` rồi lấy min theo cost.
- Mức 2: Có thể hòa — validator chấp nhận mọi dãy tối ưu.
- Mức 3: Kiểm mọi `t`.

**Bài 5**
- Mức 1: 6039 lần gọi `min_cost_general` — đủ nhanh.
- Mức 2: Ghi lại `t` có chi phí lớn nhất (lấy `t` nhỏ nhất khi hòa).
- Mức 3: In dãy phím bằng cách tìm lại dãy đạt chi phí đó.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1
r1 = Solution().minCostSetTime(0, 1, 1, 6039)
print("Bài 1:", r1); assert r1 == 5 == min_cost_brute(0, 1, 1, 6039)

# Bài 2
print("Bài 2:"); r2 = trace_two(4, 3, 2, 100); assert r2 == 15 == min_cost_brute(4, 3, 2, 100)

# Bài 3
def count_key_sequences_sol(t):
    total = 0
    for m in range(100):
        s = t - 60 * m
        if 0 <= s <= 99:
            four = f"{m:02d}{s:02d}"
            z = len(four) - len(four.lstrip("0"))
            total += z + 1
    return total

for t in range(1, 6040):
    assert count_key_sequences_sol(t) == len(BY_TIME[t])
print("Bài 3 OK (mọi t); ví dụ t=76:", count_key_sequences_sol(76), BY_TIME[76])

Bài 1: 5
Bài 2:
  t = 100: divmod -> (m, s) = (1, 40)
    ứng viên (1, 40) -> bấm '140': di 4->1(+3), bấm 1(+2), di 1->4(+3), bấm 4(+2), di 4->0(+3), bấm 0(+2) = 15
    ứng viên (0, 100): không hợp lệ
  => min = 15
Bài 3 OK (mọi t); ví dụ t=76: 5 ['76', '076', '116', '0076', '0116']


In [28]:
# Bài 4
def best_keys_sol(startAt, moveCost, pushCost, t):
    m, s = divmod(t, 60)
    opts = []
    for mm, ss in [(m, s), (m - 1, s + 60)]:
        if 0 <= mm <= 99 and 0 <= ss <= 99:
            k = f"{mm:02d}{ss:02d}".lstrip("0")
            opts.append((cost_of(k, startAt, moveCost, pushCost), k))
    return min(opts)

rnd = random.Random(4)
for t in range(1, 6040):
    st, mc, pc = rnd.randint(0, 9), rnd.randint(1, 9), rnd.randint(1, 9)
    c, k = best_keys_sol(st, mc, pc, t)
    assert interpret(k) == t and 1 <= len(k) <= 4 and c == cost_of(k, st, mc, pc) == min_cost_brute(st, mc, pc, t)
print("Bài 4 OK; ví dụ:", best_keys_sol(1, 2, 1, 600), best_keys_sol(0, 1, 2, 76))

# Bài 5
def hardest_time_on_keypad_sol(startAt):
    dist = manhattan(1)
    best_t, best_c = None, -1
    for t in range(1, 6040):
        c = min_cost_general(startAt, t, dist, 1)
        if c > best_c:
            best_t, best_c = t, c
    keys = [k for k in BY_TIME[best_t]
            if sum(dist(p, int(ch)) for p, ch in zip([startAt] + [int(x) for x in k[:-1]], k)) + len(k) == best_c]
    return best_t, best_c, keys

bt, bc, bk = hardest_time_on_keypad_sol(5)
assert bc == brute_general(5, bt, manhattan(1), 1)
print(f"Bài 5: start=5, t khó nhất = {bt} giây, chi phí {bc}, dãy tối ưu {bk}")

Bài 4 OK; ví dụ: (6, '1000') (6, '76')
Bài 5: start=5, t khó nhất = 640 giây, chi phí 16, dãy tối ưu ['1040']


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
def cost(m, s):
    if not (0 <= m <= 99 and 0 <= s <= 99): return inf
    total, prev = 0, startAt
    for ch in f"{m:02d}{s:02d}".lstrip("0"):
        d = int(ch)
        total += pushCost + (moveCost if d != prev else 0)
        prev = d
    return total
m, s = divmod(t, 60)
return min(cost(m, s), cost(m - 1, s + 60))
```

**Checklist:**
- [ ] Xét **cả hai** ứng viên `(m, s)` và `(m − 1, s + 60)`?
- [ ] Kiểm `0 ≤ m ≤ 99` và `0 ≤ s ≤ 99`?
- [ ] `lstrip("0")`, không phải `replace("0", "")`?
- [ ] `moveCost` chỉ khi đổi chữ số (bắt đầu từ `startAt`)?
- [ ] Ví dụ tự dựng: kỳ vọng lấy từ vét cạn, không tính nhẩm (xem mục 12)?
- [ ] Chữ ký đúng: `def minCostSetTime(self, startAt: int, moveCost: int, pushCost: int, targetSeconds: int) -> int`?